<a href="https://colab.research.google.com/github/MatthewFaiello/MLsessions/blob/main/ISEA_Week5_ML_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Your Task: Predict Student Success

The purpose of this HW is to get you hands on with real data trying out the modelling techniques we talked about.

You are free to use gen-ai with this project to help with the coding (of course, you don't have to!). [Hands on Machine Learning](https://www.oreilly.com/library/view/hands-on-machine-learning/9781098125967/) is also a great resource.

Your code needs to run, but I want you to focus less on the specifics of the code and more on understanding the component steps that go into building and validating a model. Creating code is now pretty easy, creating a "good" model is hard.

For this exercise we will use open data on student dropout from Portugal. Full documentation is available [here](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

M.V.Martins, D. Tolledo, J. Machado, L. M.T. Baptista, V.Realinho. (2021) "Early prediction of student’s performance in higher education: a case study" Trends and Applications in Information Systems and Technologies, vol.1, in Advances in Intelligent Systems and Computing series. Springer. DOI: 10.1007/978-3-030-72657-7_16

You will turn in on the class website a google slide deck that has:
1. A cover slide contianing your name (and all group member names if working together) and a link to your colab (**Create slide 1 now**)
2. 3 slides answering the questions below - they are clearly indicated as you go through the colab notebook.


# Get the data

Here I provide some code to get the data for you

In [119]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

import pandas as pd

from sklearn.model_selection import train_test_split

import numpy as np

In [120]:
# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

# data (as pandas dataframes)
X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets
df = df = X.join(y)

# metadata
print(predict_students_dropout_and_academic_success.metadata)

# variable information
print(predict_students_dropout_and_academic_success.variables)

{'uci_id': 697, 'name': "Predict Students' Dropout and Academic Success", 'repository_url': 'https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success', 'data_url': 'https://archive.ics.uci.edu/static/public/697/data.csv', 'abstract': "A dataset created from a higher education institution (acquired from several disjoint databases) related to students enrolled in different undergraduate degrees, such as agronomy, design, education, nursing, journalism, management, social service, and technologies.\nThe dataset includes information known at the time of student enrollment (academic path, demographics, and social-economic factors) and the students' academic performance at the end of the first and second semesters. \nThe data is used to build classification models to predict students' dropout and academic sucess. The problem is formulated as a three category classification task, in which there is a strong imbalance towards one of the classes.", 'area': 'Social Sc

# 1 Data Checking

- Look at your outcome variable - any cases to exclude?
- Determine the base-rate accuracy for a naive model
- Create Test and Training Sets
- Look at distributions of x variables, look up meta data, decide which to include

At the end of this section you should have
`x_train`, `x_text`, `y_train`, `y_test`
And an estimate of the base rate accuracy.

In [134]:
# check target distribution
pd.DataFrame({"count": df["Target"].value_counts(dropna=False),
              "percent": df["Target"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# recode so target is either "Graduate" or "Other"
df["Target_"] = df["Target"].replace({"Dropout": 0, "Enrolled": 0, "Graduate": 1})

# target base-rate
pd.DataFrame({"count": df["Target_"].value_counts(dropna=False),
              "percent": df["Target_"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# check features
metaData = predict_students_dropout_and_academic_success.variables; metaData
num = df.select_dtypes(include="number")
pd.DataFrame({"mean": num.mean(),
              "median": num.median(),
              "mode": num.mode(dropna=True).iloc[0]})

# select features
range_cols = df.iloc[:, np.r_[4, 6, 12:32, 37]].columns
df_subset = df.loc[:, range_cols]; df_subset

# stratified test and train sets
X = df_subset.drop(columns=["Target_"])
y = df_subset["Target_"]

# split (stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)
# check stratification
pd.DataFrame({"y_train": y_train.value_counts(normalize=True, dropna=False).mul(100).round(2),
              "y_test": y_test.value_counts(normalize=True, dropna=False).mul(100).round(2)})

/tmp/ipython-input-303/1281890171.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Target_"] = df["Target"].replace({"Dropout": 0, "Enrolled": 0, "Graduate": 1})


,y_train,y_test
Target_,,
0,50.07,50.06
1,49.93,49.94


# 2 Train a Model
* Pick one of the models we discussed today and train it.
* Report its accuracy and print a confusion matrix.
   * How much better is your model than the base rate?
   * How does accuracy on the train set compare to accuracy on the test set?
   * **Report Slide 2: Include an image of the confusion matrix, the base rate accuracy, train-set accuracy and test-set accuracy**

In [158]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# L1 logistic regression (lasso)
model = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(penalty="l1", solver="saga", max_iter=5000, C=1.0))])

model.fit(X_train, y_train)

# accuracy
train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test, y_test)

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

# confusion matrix (test)
pred_test = model.predict(X_test)
labels = model.named_steps["clf"].classes_
cm = confusion_matrix(y_test, pred_test, labels=labels)
print("\nConfusion matrix (test):")
print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))

# base rate
base_rate = y_test.mean()
improvement_pp = (test_acc - base_rate) * 100
print(f"\nBase rate (test majority class): {base_rate*100:.2f}%")
print(f"Improvement over base rate: {improvement_pp:.2f} percentage points")

# train vs test comparison
gap_pp = (train_acc - test_acc) * 100
print(f"\nTrain - Test accuracy gap: {gap_pp:.2f} percentage points")


Base rate (test majority class): 49.94%
Improvement over base rate: 32.66 percentage points


# 3 Train a Different Model
* Repeat all the steps in 2, but use a different model
* In addition, compare the accuracy of 1 and 2
* **Report Slide 3: Model 2 confusion matrix, train-set accuracy and test-set accuracy. Comparison Model 1 and Model 2 accuracy**

# 4 Reflection
* **Type responses on Slide 4**
* Contextualizing accuracy - think about different use cases for your model, which ones would you feel its accurate enough to use for? I only asked you to look at overall accuracy, is that good enough?
* Contextualizing features - think about these same use cases, are the prediction features you included appropriate for these uses?
* Generalizability - again thinking about your features, could you use this model in other educational contexts? How hard would it be to get that same data? Are there issues with it generalizing over time and location?

# 5 Extra Credit
* Consider ensembling your two models. Does that perform better?
* Check accuracy for different subgroups